# Notebook 07 — Table 4: HML Double-Sort Returns

**Data needed:** `data/firm_panel.csv`  
Columns required: `year_month, permno, cluster, ret, me, [45 chars]`

**Output:** `Table4_HML_DoubleSort.{csv,tex}`

For each of 45 characteristics × each cluster, this notebook:
1. Within each (date, cluster), splits firms at the median of the characteristic
2. Computes VW return spread (High − Low)
3. Reports means with NW t-statistics for L1-L5 and L46-L50
4. Counts statistically significant positive/negative clusters (#PosSig, #NegSig, PropSig)

In [8]:
import os, sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO_ROOT)
from utils.data_utils import (load_cluster_panel, load_cluster_ranking, pivot_and_rank,
                               load_firm_panel, nw_tstat, stars, save_table,
                               DATA_DIR, CHARS_45)
NW_LAGS = 3

In [13]:
# ── Load cluster panel + ranking ──────────────────────────────────
ranking        = load_cluster_ranking()
df_cl          = load_cluster_panel(K=50, lam=1_000_000)
cr_r, rank_map = pivot_and_rank(df_cl, lam=1_000_000, ranking_df=ranking)

# ── Load pre-computed HML panel ───────────────────────────────────
HML_PATH = "/ssd1/songjiangliu/shared/asset_clustering/Code/hml_outputs/hml_panel_K_50_lambda_1000000.csv"

hml = pd.read_csv(HML_PATH)
hml["year_month"] = pd.to_datetime(hml["year_month"], format="%Y-%m").dt.strftime("%Y-%m")
hml["cluster_ranked"] = hml["cluster"].map(rank_map)
hml = hml.dropna(subset=["cluster_ranked"])

char_cols = sorted(hml["feature"].unique().tolist())

print(f"HML panel: {hml.shape}")
print(f"Columns  : {list(hml.columns)}")
print(f"Features : {len(char_cols)}  |  {char_cols[:5]}...")
print(f"Period   : {hml['year_month'].min()}  ->  {hml['year_month'].max()}")
print(f"Clusters : {hml['cluster_ranked'].nunique()}")

HML panel: (1102456, 16)
Columns  : ['year_month', 'cluster', 'feature', 'N_H', 'N_L', 'R_w_H', 'R_w_L', 'HML_return', 't_within_return', 'p_within_return', 'X_w_H', 'X_w_L', 'HML_feature', 't_within_feat', 'p_within_feat', 'cluster_ranked']
Features : 45  |  ['A2ME', 'AC', 'AT', 'ATO', 'B2M']...
Period   : 1977-01  ->  2019-12
Clusters : 50


In [15]:
# ── Table 4: HML double-sort using pre-computed HML panel ─────────
TARGET_CLUSTERS = [f"L{i:02d}" for i in range(1, 6)] + \
                  [f"L{i:02d}" for i in range(46, 51)]
ALL_CLUSTERS    = [f"L{i:02d}" for i in range(1, 51)]

hml_results = {}
sig_counts  = {}

print("Computing Table 4 statistics...")

for feat, feat_grp in hml.groupby("feature"):
    # ── NW t-stat per cluster ──────────────────────────────────────
    pos_sig = neg_sig = 0
    cell_results = {}

    for cl in ALL_CLUSTERS:
        sub = feat_grp[feat_grp["cluster_ranked"] == cl]["HML_return"].dropna()
        if len(sub) < 12:
            cell_results[cl] = np.nan
            continue
        m, se, t = nw_tstat(sub, lags=NW_LAGS)
        cell_results[cl] = m
        if not pd.isna(t):
            if t >=  1.96: pos_sig += 1
            if t <= -1.96: neg_sig += 1

    # ── Format target clusters with significance stars ─────────────
    disp = {}
    for cl in TARGET_CLUSTERS:
        m = cell_results.get(cl, np.nan)
        if pd.isna(m):
            disp[cl] = ""
            continue
        sub = feat_grp[feat_grp["cluster_ranked"] == cl]["HML_return"].dropna()
        _, _, t = nw_tstat(sub, lags=NW_LAGS)
        disp[cl] = f"{m:.4f}{stars(t)}"

    hml_results[feat] = disp
    sig_counts[feat]  = {
        "#PosSig": pos_sig,
        "#NegSig": neg_sig,
        "PropSig": round((pos_sig + neg_sig) / 50, 2),
    }
    print(f"  {feat}... done")

hml_df = pd.DataFrame(hml_results).T[TARGET_CLUSTERS]
sig_df = pd.DataFrame(sig_counts).T
table4 = pd.concat([hml_df, sig_df], axis=1)
table4.index.name = "Feature"

Computing Table 4 statistics...
  A2ME... done
  AC... done
  AT... done
  ATO... done
  B2M... done
  BETA_d... done
  BETA_m... done
  C2A... done
  CF2B... done
  CF2P... done
  CTO... done
  D2A... done
  D2P... done
  DPI2A... done
  E2P... done
  FC2Y... done
  HIGH52... done
  INV... done
  IdioVol... done
  LEV... done
  ME... done
  NI... done
  NOA... done
  OA... done
  OL... done
  OP... done
  PCM... done
  PM... done
  PROF... done
  Q... done
  R12_2... done
  R12_7... done
  R2_1... done
  R36_13... done
  R60_13... done
  RNA... done
  ROA... done
  ROE... done
  RVAR... done
  S2P... done
  SGA2S... done
  SPREAD... done
  SUV... done
  TURN... done
  VAR... done


In [ ]:
print("\n=== Table 4 (first 5 rows) ===")
print(table4.head())


In [ ]:
save_table(table4, "Table4_HML_DoubleSort")
print("Table 4 saved.")